In [ ]:
# notebook version of the project.

In [ ]:
from typing import TypedDict,Optional,NotRequired,Dict,Annotated
from langgraph.graph import StateGraph, START,END
from pprint import pprint
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
from langchain_core.tools import tool
from PyPDF2 import PdfReader
from docx import Document
import glob
from concurrent.futures import ThreadPoolExecutor
import shutil
import operator
import csv
from langchain_groq import ChatGroq 
#from langchain_ollama import ChatGroq
load_dotenv()

In [ ]:
def update_resumes(existing: list, new: list) -> list:
    # Create a dictionary for quick lookup by filename
    combined = {item['filename']: item for item in existing}
    for item in new:
        combined[item['filename']] = item # Overwrites if filename exists
    return list(combined.values())

In [ ]:
def merge_unique_errors(left: List[str], right: List[str]) -> List[str]:
    # Only add the error if it's not already in the list
    return list(set(left + right))

In [ ]:
class agentstate(TypedDict):
    job_description : NotRequired[str]
    jd_file_path : str
    jd_scoring_rubric : NotRequired[dict]
    _internal_counter:NotRequired[int]
    _jd_rubric_error: Annotated[list[str], operator.add]
    resume_folder_path: str
    parsed_resumes: Annotated[list[dict], update_resumes]
    current_resume: Optional[Dict]
    scored_resumes: Annotated[list[dict], update_resumes]
    errors: Annotated[list[str], merge_unique_errors]
    current_evaluation: dict
    current_candidate: Optional[Dict]
    current_status: Optional[str]
    structured_resumes: Annotated[list[dict], update_resumes]

In [ ]:
# writing the job description node

def jdloader_node(state: agentstate) -> agentstate:
    '''this node loads the job description which can be taken as benchmark for scoring'''
    if "errors" not in state:
        state["errors"] = []
    if "_jd_rubric_error" not in state:
        state["_jd_rubric_error"] = []

    path = state.get('jd_file_path', "jd.txt")

    # 1. Logic to handle folder vs. file
    if os.path.isdir(path):
        # If it's a folder, append 'jd.txt' to the path
        final_path = os.path.join(path, "jd.txt")
        print(f"Directory detected. Looking for: {final_path}")
    else:
        final_path = path

    print(f"Loading Job Description from {final_path}")
    try:
        with open(final_path, 'r', encoding='utf-8') as jdtxt:
            content = jdtxt.read()
            state["job_description"] = content
        if not content.strip():
            state['errors'].append('Job Description file is empty')
            return state

    except FileNotFoundError:
        state['errors'].append(f"file not found at {final_path}")
        return state
    except Exception as e:
        state['errors'].append(f"Error reading file: {str(e)}")
        return state
    return state
        
    

    

In [ ]:

class Requirement(BaseModel):
    skill: str = Field(description="The specific skill or qualification required (e.g., Python, AWS, 3+ years experience).")
    importance: str = Field(description="Level of importance: 'Required' or 'Preferred'.")
    weightage: int = Field(description="Score value from 1 to 10 based on importance.")

class JDRubric(BaseModel):
    job_title: str
    technical_skills: List[Requirement]
    soft_skills: List[Requirement]
    experience_requirements: List[Requirement]
    

In [ ]:
# Or your preferred provider

def rubric_generator_node(state: agentstate) -> agentstate:
    """Analyzes the JD and creates a structured scoring rubric."""
    current_count = state.get("_internal_counter", 0)
    state["_internal_counter"] = current_count + 1
    print(f"---Execution Attempt: {state['_internal_counter']}---")
    
    # Initialize the LLM with structured output
    llm = ChatGroq(model="openai/gpt-oss-safeguard-20b")
    structured_llm = llm.with_structured_output(JDRubric)
    
    jd_text = state["job_description"]
    
    if not jd_text:
        state['errors'].append("No Job Description found to generate rubric.")
        return state

    prompt = f"""
    Analyze the following Job Description and create a structured scoring rubric.
    Break down requirements into technical skills, soft skills, and experience.
    Assign a weightage (1-10) to each item based on how central it is to the role.
    
    Job Description:
    {jd_text}
    """
    
    try:
        # Generate the rubric
        rubric_response = structured_llm.invoke(prompt)
        
        # Convert Pydantic object to dict for the state
        state["jd_scoring_rubric"] = rubric_response.model_dump()
        print("Successfully generated scoring rubric.")
        
    except Exception as e:
        state['errors'].append(f"Error generating rubric: {str(e)}")
        
    return state

In [ ]:
def validation_node(state: agentstate) -> agentstate:
    """Checks if the rubric is high-quality and complete."""
    rubric = state.get("jd_scoring_rubric", {})
    
    # Check if the rubric is missing major sections
    required_keys = ['job_title','technical_skills','soft_skills','experience_requirements']
    missing = [key for key in required_keys if not rubric.get(key)]
    
    if missing:
        error_msg = f"Validation Failed: Missing rubric sections: {', '.join(missing)}"
        state['_jd_rubric_error'].append(error_msg)
        print(f" {error_msg}")
    else:
        print("Validation Passed: Rubric is complete.")
        
    return state

In [ ]:
def should_continue(state: agentstate):
    """Router that treats the counter as Read-Only."""
    MAX_RETRIES = 3
    
    # We retrieve the count but NEVER modify it here
    count = state.get("_internal_counter", 0)
    
    if not state.get("_jd_rubric_error"):
        return "continue"
    
    if count < MAX_RETRIES:
        return "retry"
    
    return END

In [ ]:
@tool
def resume_parser_tool(file_path:str)-> str:
    """Parses the content of the resume file given its local path.
    Supports pdf, word file and text formats"""
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext==".pdf":
            reader = PdfReader(file_path)
            return " ".join([page.extract_text() for page in reader.pages if page.extract_text()])
        elif ext=='.txt':
            with open(file_path,'r') as f:
                reader = f.read()
                return reader
        elif ext == ".docx":
                doc = Document(file_path)
                return " ".join([para.text for para in doc.paragraphs])
        else: 
            return f'Unsupported file type {ext}. Supported files types are PDF, WORD file, TEXT'
    
    except Exception as e:
        return f"Error parsing {os.path.basename(file_path)}: {str(e)}"


In [ ]:
def resume_ingestion_node(state: agentstate) -> agentstate:
    """Phase 2: Parse exactly one resume per run."""

    # Ensure 'errors' key exists in state
    if "errors" not in state:
        state["errors"] = []

    resume_folder_path = state.get('resume_folder_path')

    # Check if path does NOT exist
    if not resume_folder_path or not os.path.exists(resume_folder_path):
        state['errors'].append(f'Folder not found at {resume_folder_path}.')
        return state

    # Gather files
    files = []
    for ext in ["*.pdf", "*.txt", "*.docx"]:
        files.extend(glob.glob(os.path.join(resume_folder_path, ext)))

    if not files:
        state['errors'].append("No resumes found in the specified folder.")
        return state

    # Process ONE resume at a time
    file_path = files[0]
    filename = os.path.basename(file_path)
    print(f'--- Ingesting 1 resume: {filename} ---')

    text = resume_parser_tool.invoke(file_path)

    # Handle parser errors
    if "Error parsing" in text or "Unsupported" in text:
        state["errors"].append(text)
        error_dir = os.path.join(resume_folder_path, "processing_errors")
        os.makedirs(error_dir, exist_ok=True)
        try:
            shutil.move(file_path, os.path.join(error_dir, filename))
        except Exception as e:
            state["errors"].append(f"Failed to archive bad input {filename}: {str(e)}")
        return state

    # Setup output directory
    output_directory = os.path.join(resume_folder_path, "parsed_output")
    os.makedirs(output_directory, exist_ok=True)

    # Store in state
    state["parsed_resumes"] = [{
        "filename": filename,
        "content": text
    }]
    state["current_resume"] = {
        "filename": filename,
        "content": text
    }

    # Save to local folder as .txt
    save_path = os.path.join(output_directory, f"{filename}.txt")
    with open(save_path, 'w', encoding='utf-8') as f:
        f.write(text)

    # Move original file so it won't be reprocessed
    input_archive = os.path.join(resume_folder_path, "input_archive")
    os.makedirs(input_archive, exist_ok=True)
    try:
        shutil.move(file_path, os.path.join(input_archive, filename))
    except Exception as e:
        state["errors"].append(f"Failed to archive input {filename}: {str(e)}")

    print("Successfully ingested 1 resume.")
    return state

In [ ]:
class JobEntry(BaseModel):
    """Sub-model for a single professional role"""
    job_title: str = Field(description="The job title held by the candidate")
    experience_details: str = Field(description="Full description of roles, duties, and metrics for this specific job")

class ExtractedResume(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate")
    email: Optional[str] = Field(description="Contact Email Address")
    total_year_experience: float = Field(description="Total years of professional experience")
    
    # This creates the { "experience_1": {...}, "experience_2": {...} } structure
    experience: Dict[str, JobEntry] = Field(
        description="A dictionary where keys are 'experience_1', 'experience_2', etc., and values contain the job title and full experience text."
    )
    
    technical_skills: List[str] = Field(description="List of technical skills")
    projects: List[str] = Field(description="List all the projects with title and description")
    education: List[str] = Field(description="List of degrees or certifications")

In [ ]:
def structured_resume_extractor_node(state: agentstate) -> agentstate:
    """Phase 2, Step 5: Physically fetches one resume from the folder at a time."""
    
    # 1. CRITICAL: Initialize lists in state if they don't exist
    if "structured_resumes" not in state or state["structured_resumes"] is None:
        state["structured_resumes"] = []
    if "errors" not in state:
        state["errors"] = []
    
    resume_folder_path = state.get("resume_folder_path")
    parsed_dir = os.path.join(resume_folder_path, "parsed_output")
    # Using 'processed_archive' to stay consistent with Phase 2 naming
    processed_dir = os.path.join(parsed_dir, "processed_archive")
    
    os.makedirs(processed_dir, exist_ok=True)
    
    # 2. Fetch files from disk
    text_files = glob.glob(os.path.join(parsed_dir, "*.txt"))
    
    if not text_files:
        print('--- All resumes have been processed. ---')
        return state
    
    current_file_path = text_files[0]
    filename = os.path.basename(current_file_path)
    
    print(f"--- Fetching from disk: {filename} ({len(text_files)-1} remaining) ---")
    
    try:
        with open(current_file_path, 'r', encoding='utf-8') as f:
            content = f.read()
            

        llm = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
        # Force JSON schema output to prevent tool-call hallucinations
        structured_llm = llm.with_structured_output(ExtractedResume, method="json_schema")
        
        prompt = f"""
            Extract professional information from the following resume text.
            For the 'experience' dictionary:
            1. Use keys like 'experience_1', 'experience_2', etc., in chronological order (most recent first).
            2. For each entry, include the 'job_title' and a detailed 'experience_details' string containing all duties and metrics for that role.

            Resume Text:
            {content}
            """
        structured_data = structured_llm.invoke(prompt)
        
        # 4. Update State
        state['structured_resumes'].append({
            "filename": filename, 
            "data": structured_data.model_dump()
        })
        
        # 5. Move file to mark as processed
        shutil.move(current_file_path, os.path.join(processed_dir, filename))
        print(f"Successfully processed and archived: {filename}")
        
    except Exception as e:
        error_msg = f"Extraction failed for {filename}: {str(e)}"
        print(f"Error: {error_msg}")
        state['errors'].append(error_msg)
        
        # Move to error folder so it doesn't block the loop
        error_dir = os.path.join(parsed_dir, "processing_errors")
        os.makedirs(error_dir, exist_ok=True)
        shutil.move(current_file_path, os.path.join(error_dir, filename))

    # 6. ALWAYS return state
    return state

In [ ]:
def check_folder_queue_node(state:agentstate):
    '''Router node that checks for remaining input resumes.'''
    resume_folder_path = state.get("resume_folder_path")

    remaining_files = []
    for ext in ["*.pdf", "*.txt", "*.docx"]:
        remaining_files.extend(glob.glob(os.path.join(resume_folder_path, ext)))

    if len(remaining_files) > 0:
        return "next_resume"
    return "complete"

In [ ]:
class SkillMatch(BaseModel):
    skill_name:str
    match_found:bool
    score_assigned:int = Field(description='Score from 0 to maximum weightage that is defined in rubric')
    reasoning:str = Field(description='Explanation for the score given based on the resume evidence')
    
class CandidateEvaluation(BaseModel):
    candidate_name:str
    technical_score:list[SkillMatch]
    soft_skill_score:list[SkillMatch]
    experience_score: list[SkillMatch]
    total_weighted_score:float
    gap_analysis: list[str] = Field(description='List of all the critical requirements that are missing in the resume')

In [ ]:
def rubric_matcher_node(state:agentstate)->agentstate:
    '''In this node the resumes are matched and scored from the rubric'''
    if "scored_resumes" not in state or state["scored_resumes"] is None:
        state["scored_resumes"] = []
    rubric = state.get('jd_scoring_rubric')
    structured_data_list = state.get('structured_resumes',[])
    llm = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
    
    structured_llm = llm.with_structured_output(CandidateEvaluation)
    print(f'---Evaluating {len(structured_data_list)} candidates against rubric ---')
    
    for resume_entry in structured_data_list:
        candidate_data = resume_entry['data']
        prompt = f'''
        You are an experienced HR professional and talent evaluator.
        Evaluate the candidate '{candidate_data['candidate_name']}' based on the provided Scoring Rubric.
        
        Scoring Rubric:
        {rubric}
        
        Candidate Structured Data:
        {candidate_data}
        
        Instructions:
        1. For each requirement in the rubric, check if the candidate possesses it.
        2. Assign a score for each item up to its defined weightage.
        3. Calculate a final weighted score.
        4. List specific gaps where the candidate does not meet 'Required' importance.
        '''
        try:
            evaluation = structured_llm.invoke(prompt)
            state["scored_resumes"].append({
                "filename": resume_entry["filename"],
                "evaluation": evaluation.model_dump()
            })
            print(f"Evaluation complete for: {candidate_data['candidate_name']}")
        except Exception as e:
            state['errors'].append(f"Evaluation failed for {resume_entry['filename']}: {str(e)}")

    return state


In [ ]:
def rubric_matcher_node(state: agentstate) -> agentstate:
    """Evaluates ONLY the single most recently structured resume."""
    if "scored_resumes" not in state or state["scored_resumes"] is None:
        state["scored_resumes"] = []
        
    rubric = state.get('jd_scoring_rubric')
    
    # CRITICAL CHANGE: Only pick the last resume added to the list
    if not state.get('structured_resumes'):
        state['errors'].append("No structured resume found to match.")
        return state
        
    resume_entry = state['structured_resumes'][-1] 
    candidate_data = resume_entry['data']
    
    llm = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
    # Force JSON schema output to prevent tool-call hallucinations
    structured_llm = llm.with_structured_output(CandidateEvaluation, method="json_schema")
    
    prompt = f'''
        You are an experienced HR professional and talent evaluator.
        Evaluate the candidate '{candidate_data['candidate_name']}' based on the provided Scoring Rubric.
        
        Scoring Rubric:
        {rubric}
        
        Candidate Structured Data:
        {candidate_data}
        
        Instructions:
        1. For each requirement in the rubric, check if the candidate possesses it.
        2. Assign a score for each item up to its defined weightage.
        3. Calculate a final weighted score.
        4. List specific gaps where the candidate does not meet 'Required' importance.
        '''

    
    try:
        evaluation = structured_llm.invoke(prompt)
        eval_dict = evaluation.model_dump()
        
        # Store evaluation in state
        state["scored_resumes"].append({
            "filename": resume_entry["filename"],
            "evaluation": eval_dict
        })
        
        # Also set 'current_evaluation' for the Sorter to read easily
        state["current_evaluation"] = {
            "filename": resume_entry["filename"],
            "score": eval_dict["total_weighted_score"]
        }
        
        # Store candidate info for spreadsheet update
        state["current_candidate"] = {
            "filename": resume_entry["filename"],
            "candidate_name": candidate_data.get("candidate_name", ""),
            "email": candidate_data.get("email") or ""
        }
        
        print(f"Scored {candidate_data['candidate_name']}: {eval_dict['total_weighted_score']}")
    except Exception as e:
        state['errors'].append(f"Matching failed: {str(e)}")
        
    return state

In [ ]:
def resume_sorter_node(state: agentstate) -> agentstate:
    """Physically moves the file and keeps the state-in, state-out pattern."""
    eval_data = state.get("current_evaluation")
    if not eval_data:
        return state
        
    score = eval_data["score"]
    filename = eval_data["filename"]
    base_path = state.get("resume_folder_path")
    
    # Determine the folder
    if score >= 85:
        target = "Accepted"
    elif 50 <= score < 85:
        target = "Review"
    else:
        target = "Rejected"

    # Store status for spreadsheet update
    state["current_status"] = target
        
    # Physical move: from processed_archive to final destination
    src = os.path.join(base_path, "parsed_output", "processed_archive", filename)
    dest_dir = os.path.join(base_path, target)
    os.makedirs(dest_dir, exist_ok=True)
    
    try:
        shutil.move(src, os.path.join(dest_dir, filename))
        print(f"SORTED: {filename} moved to {target}")
    except Exception as e:
        state["errors"].append(f"Physical move failed: {str(e)}")
        
    return state

In [ ]:
def spreadsheet_update_node(state: agentstate) -> agentstate:
    """Updates a per-JD CSV score sheet with the latest candidate."""
    candidate = state.get("current_candidate") or {}
    eval_data = state.get("current_evaluation") or {}
    status = state.get("current_status")

    if not candidate or status is None or not eval_data:
        return state

    full_name = (candidate.get("candidate_name") or "").strip()
    name_parts = [p for p in full_name.split(" ") if p]
    first_name = name_parts[0] if name_parts else ""
    last_name = name_parts[-1] if len(name_parts) > 1 else ""

    email = candidate.get("email") or ""
    score = eval_data.get("score", "")
    filename = candidate.get("filename") or ""

    jd_path = state.get("jd_file_path", "jd.txt")
    if os.path.isdir(jd_path):
        jd_path = os.path.join(jd_path, "jd.txt")
    jd_id = os.path.splitext(os.path.basename(jd_path))[0] or "jd"

    resume_folder_path = state.get("resume_folder_path", ".")
    sheet_dir = os.path.join(resume_folder_path, "score_sheets")
    os.makedirs(sheet_dir, exist_ok=True)
    sheet_path = os.path.join(sheet_dir, f"{jd_id}_scores.csv")

    headers = ["First Name", "Last Name", "Email", "Score", "Status", "Filename"]
    rows = []

    if os.path.exists(sheet_path):
        with open(sheet_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)

    updated = False
    for row in rows:
        same_file = filename and row.get("Filename") == filename
        same_email = email and row.get("Email") == email
        if same_file or same_email:
            row.update({
                "First Name": first_name,
                "Last Name": last_name,
                "Email": email,
                "Score": score,
                "Status": status,
                "Filename": filename
            })
            updated = True
            break

    if not updated:
        rows.append({
            "First Name": first_name,
            "Last Name": last_name,
            "Email": email,
            "Score": score,
            "Status": status,
            "Filename": filename
        })

    with open(sheet_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Updated score sheet: {sheet_path}")
    return state

In [ ]:
graph = StateGraph(agentstate)

graph.add_node('jdloader_node',jdloader_node)
graph.add_node('rubric_generator_node', rubric_generator_node)
graph.add_node('validation_node',validation_node)
graph.add_node('resume_ingestion_node', resume_ingestion_node)
graph.add_node('structured_resume_extractor_node', structured_resume_extractor_node)
graph.add_node('rubric_matcher_node', rubric_matcher_node)
graph.add_node('resume_sorter_node', resume_sorter_node)
graph.add_node('spreadsheet_update_node', spreadsheet_update_node)


graph.add_edge(START,'jdloader_node')
graph.add_edge('jdloader_node', 'rubric_generator_node')
graph.add_edge('rubric_generator_node','validation_node')
graph.add_conditional_edges(
    "validation_node",
    should_continue,
    {
        "continue": 'resume_ingestion_node', 
        "retry": "rubric_generator_node",  
        END: END
    }
)
graph.add_edge('resume_ingestion_node', 'structured_resume_extractor_node')
graph.add_edge('structured_resume_extractor_node', 'rubric_matcher_node')
graph.add_edge('rubric_matcher_node', 'resume_sorter_node')
graph.add_edge('resume_sorter_node', 'spreadsheet_update_node')

graph.add_conditional_edges(
    "spreadsheet_update_node", # Check AFTER updating the sheet
    check_folder_queue_node,
    {
        "next_resume": "resume_ingestion_node", # Loop back for the next input
        "complete": END # Stop when input folder is empty
    }
)
app = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result = app.invoke({"jd_file_path":"E:\\Project\\dd\\jd.txt",'resume_folder_path':"E:\\Project\\resumes","errors": []})

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
pprint(result.keys())

In [ ]:
pprint(result['_jd_rubric_error'])

In [ ]:
pprint(result['scored_resumes'])

In [ ]:
import json

# result = app.invoke(...) was already called in your notebook


with open("final_state_output.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)

print("Scores have been saved to final_state_output.json")